# Carnage-V1 on Colab (v3-style)

1v1 Rocket League self-play trainer (GigaLearnCPP + RLGymCPP), Phase-1 reward stack, GPU training.

**How to run:**
1. Ensure **Runtime -> Change runtime type -> T4 GPU** is selected, then restart.
2. Run cells top to bottom.
3. Cell 2 mounts your Drive (click the auth link once). Checkpoints auto-backup to `My Drive/Carnage-V1/checkpoints/` from the start.
4. Optional: upload the converted replay binary to `My Drive/Carnage-V1/serialized_replays.bin`.
5. Use the last cell (STOP) to stop training cleanly - it saves a checkpoint first.

Source: `https://github.com/vfxjamer/Carnage-V1.git`

In [1]:
# cell - 1
# 1. Clone source (self-contained: src + CMakeLists + collision_meshes + GigaLearnCPP thirdparty)
import os, subprocess, shutil

ROOT = "/content/Carnage-V1"
REPO = "https://github.com/vfxjamer/Carnage-V1.git"

if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull"], check=False)

print("ROOT:", ROOT)
print("contents:", sorted(os.listdir(ROOT)))

ROOT: /content/Carnage-V1
contents: ['.git', '.gitignore', 'CMakeLists.txt', 'Carnage_colab.ipynb', 'colab_setup.sh', 'collision_meshes', 'scratchpad.ipynb', 'src', 'thirdparty']


In [2]:
# cell - 2
# 2. Mount Google Drive EARLY (so checkpoints back up from the start) + restore latest checkpoint.
import os, shutil, glob

if not os.path.isdir("/content/drive"):
    print("mounting Drive (complete the auth popup in the browser)...", flush=True)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("WARN: drive mount failed:", e, "- backups will retry automatically", flush=True)
else:
    print("drive already mounted", flush=True)

DRIVE_CKPT = "/content/drive/MyDrive/Carnage-V1/checkpoints"
LOCAL_CKPT = "/content/Carnage-V1/build/checkpoints"
DRIVE_REPLAY = "/content/drive/MyDrive/Carnage-V1/serialized_replays.bin"
LOCAL_REPLAY = "/content/Carnage-V1/build/serialized_replays.bin"

os.makedirs(LOCAL_CKPT, exist_ok=True)

def ts_of(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

if os.path.isdir(DRIVE_CKPT):
    drive_dirs = sorted([d for d in glob.glob(os.path.join(DRIVE_CKPT, "*")) if os.path.isdir(d)], key=ts_of)
    print(f"Drive has {len(drive_dirs)} checkpoint dirs")
    if drive_dirs:
        newest = drive_dirs[-1]
        dest = os.path.join(LOCAL_CKPT, os.path.basename(newest))
        if not os.path.isdir(dest):
            print("restoring latest checkpoint:", os.path.basename(newest))
            shutil.copytree(newest, dest)
else:
    print("no Drive checkpoints yet")

if os.path.exists(DRIVE_REPLAY) and not os.path.exists(LOCAL_REPLAY):
    print("copying replay binary from Drive")
    shutil.copy(DRIVE_REPLAY, LOCAL_REPLAY)

print("local checkpoints:", sorted(os.listdir(LOCAL_CKPT)) if os.path.isdir(LOCAL_CKPT) else [])
print("replay binary present:", os.path.exists(LOCAL_REPLAY))

mounting Drive (complete the auth popup in the browser)...
Mounted at /content/drive
Drive has 5 checkpoint dirs
restoring latest checkpoint: 270047744
local checkpoints: ['270047744']
replay binary present: False


In [3]:
# cell - 3
# 3. Sanity check: CLI flags, collision meshes, PPO config markers in the cloned source.
import os
ROOT = "/content/Carnage-V1"

def _has(path, needle, nice):
    if not os.path.exists(path):
        print("MISS:", path); return
    s = open(path).read()
    ok = needle in s
    print(("OK  " if ok else "MISSING ") + nice, "<-", os.path.relpath(path, ROOT))

main = os.path.join(ROOT, "src", "main.cpp")
_has(main, "--device", "cli --device")
_has(main, "--games", "cli --games")
_has(main, "--save-dir", "cli --save-dir")
_has(main, "--wandb", "cli --wandb")
_has(main, "NextoObs", "Nexto obs builder")
_has(main, "LayerNorm", "LayerNorm config")

meshes = os.path.join(ROOT, "collision_meshes")
print("collision_meshes dir:", os.path.isdir(meshes), "|", sorted(os.listdir(meshes)) if os.path.isdir(meshes) else "")

print("Carnage sanity check done.")

OK  cli --device <- src/main.cpp
OK  cli --games <- src/main.cpp
OK  cli --save-dir <- src/main.cpp
OK  cli --wandb <- src/main.cpp
OK  Nexto obs builder <- src/main.cpp
OK  LayerNorm config <- src/main.cpp
collision_meshes dir: True | ['desktop.ini', 'soccar']
Carnage sanity check done.


In [4]:
# cell - 4
# 2b. wandb setup: install the Python package the embedded interpreter will use, and set the API key.
# Pinned to 0.16.6 - newer versions segfault the data-logging path in this build.
# Two fixes:
#  1) wandb 0.16.6 uses np.float_/np.complex_ (removed in NumPy 2) -> re-add aliases
#  2) stale partially-loaded wandb cached in sys.modules -> purge before import
import os, subprocess, sys
WANDB_API_KEY = "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"

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_MODE"] = "online"
os.environ["CARNAGE_WANDB_GROUP"] = "Phase 1"
os.environ["CARNAGE_WANDB_RUN"] = "carnage-v1"

import numpy as np
if not hasattr(np, "float_"):
    np.float_ = np.float64
if not hasattr(np, "complex_"):
    np.complex_ = np.complex128

print("clearing stale wandb modules...")
for _m in list(sys.modules):
    if _m == "wandb" or _m.startswith("wandb."):
        del sys.modules[_m]

print("(re)installing wandb==0.16.6...")
r = subprocess.run(["pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "wandb==0.16.6"], capture_output=True, text=True)
print("pip rc:", r.returncode, (r.stderr or "")[-200:])
import wandb
print("wandb:", wandb.__version__)
print("key set:", bool(os.environ.get("WANDB_API_KEY")))

clearing stale wandb modules...
(re)installing wandb==0.16.6...
pip rc: 0 ompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.

wandb: 0.16.6
key set: True


In [5]:
# cell - 5
# 2. Install build deps (apt) + ensure GPU-enabled torch via pip if needed
import subprocess, sys, os

APT_PKGS = ["build-essential", "cmake", "git", "libpython3-dev", "pkg-config"]
print("apt update/install...")
r = subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True)
print("update rc:", r.returncode, (r.stderr or "")[-300:])
r = subprocess.run(["apt-get", "install", "-y", "-qq"] + APT_PKGS, capture_output=True, text=True)
print("install rc:", r.returncode, (r.stderr or "")[-300:])

# torch: pip's libtorch headers are used by CMake (TORCH_INSTALL_PREFIX).
# On a GPU runtime ensure CUDA-enabled torch. On CPU runtime keep whatever is there.
import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available(), "cuda build:", torch.version.cuda)
if torch.version.cuda is None:
    print("NOTE: torch is CPU-only. Training cell will require switching to GPU runtime.")

cmake_v = subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout
print(cmake_v.splitlines()[0])
print("gcc:", subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print("python dev headers:", os.path.exists("/usr/include/python3.12/Python.h") or os.path.exists("/usr/local/include/python3.12/Python.h") or os.path.exists("/usr/include/python3.11/Python.h"))

apt update/install...
update rc: 0 W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

install rc: 0 
torch: 2.11.0+cu128 cuda available: True cuda build: 12.8
cmake version 3.31.10
gcc: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
python dev headers: True


In [6]:
# cell - 6
# 3. Configure + build Carnage (Release). Override TORCH_INSTALL_PREFIX to pip's torch location.
import os, subprocess, sys

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

import torch
torch_prefix = os.path.dirname(torch.__file__)  # contains share/cmake/Torch
print("TORCH_INSTALL_PREFIX =", torch_prefix)
print("has Torch cmake:", os.path.exists(os.path.join(torch_prefix, "share", "cmake", "Torch")))

configure = [
    "cmake", "-S", ".", "-B", "build",
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DTORCH_INSTALL_PREFIX={torch_prefix}",
]
print(" ".join(configure))
r = subprocess.run(configure, capture_output=True, text=True)
print("configure rc:", r.returncode)
print((r.stdout or "")[-2500:])
print((r.stderr or "")[-1500:])

TORCH_INSTALL_PREFIX = /usr/local/lib/python3.12/dist-packages/torch
has Torch cmake: True
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DTORCH_INSTALL_PREFIX=/usr/local/lib/python3.12/dist-packages/torch
configure rc: 0
s GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Python3: /usr/local/bin/python (found version "3.12.13") found components: Interpreter
-- PyTorch CMake prefix: /usr/local/lib/python3.12/dist-packages/torch/share/cmake
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA comp

In [7]:
# cell - 7
# 4. Build (this is the long step). Uses all available cores.
import os, subprocess

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

nproc = os.cpu_count() or 2
print(f"Building with -j{nproc} ... (this can take 10-30 min)")
r = subprocess.run(["cmake", "--build", "build", "-j", str(nproc)], capture_output=True, text=True)
print("build rc:", r.returncode)
tail = (r.stdout or "")[-3000:] + (r.stderr or "")[-2000:]
print(tail)
print("---")
exe = os.path.join(ROOT, "build", "Carnage")
print("binary exists:", os.path.exists(exe), exe if os.path.exists(exe) else "")

Building with -j2 ... (this can take 10-30 min)
build rc: 0
Builders/DefaultObs.cpp.o
[ 86%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/RLGymCPP/CMakeFiles/RLGymCPP.dir/src/RLGymCPP/ObsBuilders/DefaultObsPadded.cpp.o
[ 87%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/RLGymCPP/CMakeFiles/RLGymCPP.dir/src/RLGymCPP/Rewards/ZeroSumReward.cpp.o
[ 87%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/RLGymCPP/CMakeFiles/RLGymCPP.dir/src/RLGymCPP/StateSetters/RandomState.cpp.o
[ 88%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/RLGymCPP/CMakeFiles/RLGymCPP.dir/src/RLGymCPP/ThreadPool.cpp.o
[ 89%] Linking CXX static library libRLGymCPP.a
[ 89%] Built target RLGymCPP
[ 90%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/CMakeFiles/GigaLearnCPP.dir/src/private/GigaLearnCPP/PPO/ExperienceBuffer.cpp.o
[ 90%] Building CXX object thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/CMakeFiles/GigaLearnCPP.dir/src/private/GigaLear

In [8]:
# cell - 8
import sys, platform
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import:", e)

python: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cu128 cuda: True
gpu: Tesla T4


In [9]:
# cell - 9
# 4b. System performance monitor -> wandb (GPU util/VRAM/temp, CPU, RAM, disk I/O)
# Runs in the background while training. Uses the SAME wandb run via run_id from the binary log.
import os, time, threading, subprocess, glob, re

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

import numpy as np
if not hasattr(np, "float_"):
    np.float_ = np.float64
if not hasattr(np, "complex_"):
    np.complex_ = np.complex128

def _read_run_id():
    logs = glob.glob("/content/Carnage-V1/build/*.log") + glob.glob("/content/Carnage-V1/*.log")
    for f in logs:
        try:
            txt = open(f, errors="ignore").read()
            m = re.search(r'run with ID : "([^"]+)"', txt)
            if m: return m.group(1)
        except Exception:
            pass
    return None

def _nvidia():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout.strip()
        vals = out.split(",")
        return dict(gpu_util=float(vals[0]), vram_used_mb=float(vals[1]), vram_total_mb=float(vals[2]), gpu_temp_c=float(vals[3]))
    except Exception:
        return None

def _cpu():
    try:
        import psutil
        return dict(cpu_util=psutil.cpu_percent(interval=1), ram_used_gb=round(psutil.virtual_memory().used / 1e9, 2),
                    ram_total_gb=round(psutil.virtual_memory().total / 1e9, 2))
    except Exception:
        return None

def sys_monitor(project, group):
    import wandb
    if not hasattr(np, "float_"):
        np.float_ = np.float64
    if not hasattr(np, "complex_"):
        np.complex_ = np.complex128
    while not os.path.exists("/content/Carnage-V1/build/Carnage"):
        time.sleep(10)
    run_id = None
    for _ in range(600):  # wait up to 1h for the training binary to start
        run_id = _read_run_id()
        if run_id: break
        time.sleep(6)
    if not run_id:
        print("system monitor: training never started (no run id found)")
        return
    run = wandb.init(project=project, group=group, name="system-monitor", id=run_id, resume=True)
    print(f"system monitor attached to run {run_id}")
    while True:
        d = {}
        g, c = _nvidia(), _cpu()
        if g: d.update({("sys/" + k): v for k, v in g.items()})
        if c: d.update({("sys/" + k): v for k, v in c.items()})
        if d: run.log(d)
        time.sleep(10)

t = threading.Thread(target=sys_monitor, args=("Carnage", "Phase 1"), daemon=True)
t.start()
print("system monitor thread started")

system monitor thread started


In [10]:
# cell - 10
# FIX: NumPy 2 removed np.float_/np.complex_, which wandb 0.16.6 still uses.
# 1) metric_receiver.py: prepend alias
# 2) sitecustomize.py in site-packages -> EVERY python process (including wandb's service subprocess) gets the alias
# 3) Directly patch wandb's _dtypes.py (the exact file that crashes)

ALIAS = '''
import numpy as _np
if not hasattr(_np, "float_"):
    _np.float_ = _np.float64
if not hasattr(_np, "complex_"):
    _np.complex_ = _np.complex128
'''

import os, sys, subprocess, site

# --- 2) global sitecustomize.py ---
SC = ("def _carnage_numpy_patch():\n"
      "    import numpy as _np\n"
      "    if not hasattr(_np, 'float_'):\n"
      "        _np.float_ = _np.float64\n"
      "    if not hasattr(_np, 'complex_'):\n"
      "        _np.complex_ = _np.complex128\n"
      "_carnage_numpy_patch()\n"
      "del _carnage_numpy_patch\n")
seen = set()
all_sp = list(site.getsitepackages()) + [p for p in sys.path if "site-packages" in p or "dist-packages" in p]
for sp in all_sp:
    if not sp or sp in seen:
        continue
    seen.add(sp)
    try:
        if not os.path.isdir(sp):
            print("skip (not a dir):", sp); continue
        p = os.path.join(sp, "sitecustomize.py")
        prev = open(p).read() if os.path.exists(p) else ""
        if "carnage_numpy_patch" not in prev:
            open(p, "w").write(SC)
            print("sitecustomize written:", p)
        else:
            print("sitecustomize already present:", p)
    except Exception as e:
        print("warn:", sp, e)
import importlib
if "sitecustomize" in sys.modules:
    importlib.reload(sys.modules["sitecustomize"])
else:
    import sitecustomize

# --- 1) metric_receiver.py (both copies) ---
for base in ["/content/Carnage-V1/thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/python_scripts",
             "/content/Carnage-V1/build/python_scripts"]:
    p = os.path.join(base, "metric_receiver.py")
    if not os.path.exists(p):
        print("MISS:", p); continue
    s = open(p).read()
    if "float_" not in s:
        open(p, "w").write(ALIAS + s)
        print("patched:", p)
    else:
        print("already patched:", p)

# --- 3) wandb _dtypes.py - use unique marker ---
dp = "/usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/_dtypes.py"
if os.path.exists(dp):
    s = open(dp).read()
    if "import numpy as _np" not in s:
        open(dp, "w").write(ALIAS + s)
        print("patched:", dp)
    else:
        print("already patched:", dp)

# --- verify: fresh interpreter (exactly what wandb's service subprocess does) ---
r = subprocess.run(["/usr/local/bin/python", "-c",
                    "import sys, numpy, wandb\n"
                    "print('exe:', sys.executable)\n"
                    "print('float_ present:', hasattr(numpy, 'float_'))\n"
                    "print('wandb', wandb.__version__)"],
                   capture_output=True, text=True)
print("rc:", r.returncode)
print(r.stdout[-1200:])
print("--- stderr tail ---")
print((r.stderr or "")[-1200:])

sitecustomize written: /usr/local/lib/python3.12/dist-packages/sitecustomize.py
sitecustomize written: /usr/lib/python3/dist-packages/sitecustomize.py
skip (not a dir): /usr/lib/python3.12/dist-packages
sitecustomize written: /usr/local/lib/python3.12/dist-packages/IPython/extensions/sitecustomize.py
sitecustomize written: /usr/local/lib/python3.12/dist-packages/setuptools/_vendor/sitecustomize.py
already patched: /content/Carnage-V1/thirdparty/GigaLearnCPP-Leak/GigaLearnCPP/python_scripts/metric_receiver.py
already patched: /content/Carnage-V1/build/python_scripts/metric_receiver.py
patched: /usr/local/lib/python3.12/dist-packages/wandb/sdk/data_types/_dtypes.py
rc: 0
exe: /usr/bin/python3
float_ present: True
wandb 0.16.6

--- stderr tail ---



In [ ]:
# cell - 11
# TRAINING SUPERVISOR
# - Auto-selects device: CUDA if available, otherwise CPU (slow but runnable).
# - Starts the 24/7 backup daemon + a live log tailer, then launches training.
# - The tailer streams the binary's real-time output (steps/timesteps/reports) into THIS cell.
# - If the binary crashes, it auto-restarts with backoff, resuming from the latest checkpoint.
# - W&B group/run come from env: CARNAGE_WANDB_GROUP, CARNAGE_WANDB_RUN.
# - Games: CARNAGE_GAMES env override, default 1024.
import os, sys, subprocess, torch, time, threading, shutil, glob

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected - training on CPU (will be slow)", flush=True)

BUILD = "/content/Carnage-V1/build"
LOCAL_CKPT = os.path.join(BUILD, "checkpoints")
DRIVE_CKPT = "/content/drive/MyDrive/Carnage-V1/checkpoints"
GAMES = int(os.environ.get("CARNAGE_GAMES", "1024"))

# ---- 24/7 backup daemon (guarded: one instance per kernel) ----
def _ts(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

def _backup_once():
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    local = sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT, "*")) if os.path.isdir(d)], key=_ts)
    drive = set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT, "*")))
    for d in local:
        name = os.path.basename(d)
        if name in drive:
            continue
        tmp = os.path.join(DRIVE_CKPT, name + ".tmp")
        try:
            print(f"[backup] uploading checkpoint {name} ...", flush=True)
            shutil.copytree(d, tmp)
            shutil.move(tmp, os.path.join(DRIVE_CKPT, name))
            print(f"[backup] {name} uploaded", flush=True)
        except Exception as e:
            print("[backup] err:", e, flush=True)
            shutil.rmtree(tmp, ignore_errors=True)

def _backup_loop():
    while True:
        try:
            if os.path.isdir("/content/drive"):
                _backup_once()
            else:
                print("[backup] drive not mounted, retrying in 60s", flush=True)
        except Exception as e:
            print("[backup] loop err:", e, flush=True)
        time.sleep(60)

if "_CARNAGE_BACKUP_DAEMON_" not in globals():
    globals()["_CARNAGE_BACKUP_DAEMON_"] = True
    threading.Thread(target=_backup_loop, daemon=True).start()
    print("[backup] 24/7 daemon started (new checkpoints -> Drive within 60s)", flush=True)

# ---- Live log tailer: streams train.log (binary output + history) into this cell ----
_tailer_lock = {"run": True}
if "_CARNAGE_TAILER_" not in globals():
    globals()["_CARNAGE_TAILER_"] = True
    def _tail_loop():
        path = os.path.join(BUILD, "train.log")
        while not os.path.exists(path):
            time.sleep(1)
        while True:
            try:
                with open(path, "r", errors="ignore") as f:
                    while True:
                        data = f.read()
                        if data:
                            sys.stdout.write(data)
                            sys.stdout.flush()
                        else:
                            if not _tailer_lock.get("run", True):
                                return
                            time.sleep(0.5)
            except SystemExit:
                return
            except Exception:
                time.sleep(1)
    threading.Thread(target=_tail_loop, daemon=True).start()
    print("[tailer] streaming binary output to this cell...", flush=True)

# ---- Training supervisor ----
os.chdir(BUILD)
os.makedirs(LOCAL_CKPT, exist_ok=True)

BASE_CMD = ["stdbuf", "-oL", "-eL", "./Carnage", "/content/Carnage-V1/collision_meshes",
            "--device", DEVICE, "--games", str(GAMES), "--save-dir", "checkpoints", "--wandb", "Carnage"]
print("BASE CMD:", " ".join(BASE_CMD), flush=True)
print("Games:", GAMES, flush=True)

backoff = 10
attempt = 0
with open("train.log", "a") as log:
    while True:
        attempt += 1
        t0 = time.time()
        print(f"[supervisor] launch #{attempt}: {' '.join(BASE_CMD[:6])} ...", flush=True)
        proc = subprocess.Popen(BASE_CMD, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        rc = proc.wait()
        elapsed = time.time() - t0
        if rc == 0:
            print("[supervisor] clean exit (rc=0) - stopping supervisor.", flush=True)
            break
        print(f"[supervisor] crashed rc={rc} after {elapsed:.0f}s - restarting in {backoff}s", flush=True)
        time.sleep(backoff)
        if elapsed < 60:
            backoff = min(backoff * 2, 600)
        else:
            backoff = 10
print("supervisor exited.")

GPU detected: Tesla T4
[backup] 24/7 daemon started (new checkpoints -> Drive within 60s)
[tailer] streaming binary output to this cell...
BASE CMD: stdbuf -oL -eL ./Carnage /content/Carnage-V1/collision_meshes --device cuda --games 1024 --save-dir checkpoints --wandb Carnage
Games: 1024
[supervisor] launch #1: stdbuf -oL -eL ./Carnage /content/Carnage-V1/collision_meshes --device ...
Initializing RocketSim version 2.1.1, created by ZealanL...
Loading arena meshes for soccar...
   > Loaded 80 verts and 126 tris, hash: 0x760358d3
   > Loaded 18 verts and 16 tris, hash: 0x1f8ee550
   > Loaded 483 verts and 880 tris, hash: 0xec759ebf
   > Loaded 536 verts and 983 tris, hash: 0x94fb0d5c
   > Loaded 483 verts and 880 tris, hash: 0x2811eee8
   > Loaded 18 verts and 16 tris, hash: 0x3d79d25d
   > Loaded 483 verts and 880 tris, hash: 0xa160baf9
   > Loaded 536 verts and 983 tris, hash: 0xdea07102
   > Loaded 80 verts and 126 tris, hash: 0x918f4a4e
   > Loaded 80 verts and 126 tris, hash: 0x73a

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: jam3r (jam3r-adobe). Use `wandb login --relogin` to force relogin


Streaming output truncated to the last 5000 lines.








Average Step Reward: 6.5756
Policy Entropy: 0.5453

Policy Update Magnitude: 0.1589
Critic Update Magnitude: 0.1719

Collection Steps/Second: 9,557.6299
Consumption Steps/Second: 58,363.1445
Overall Steps/Second: 8,212.7051

Collection Time: 8.1426
 - Inference Time: 0.2928
 - Env Step Time: 7.6075
Consumption Time: 1.3334
 - GAE Time: 9.125030e-04
 - PPO Learn Time: 1.2110
Collected Timesteps: 77,824
Total Timesteps: 279,226,880
Total Iterations: 5,095












Average Step Reward: 6.4728
Policy Entropy: 0.5476

Policy Update Magnitude: 0.1602
Critic Update Magnitude: 0.1966

Collection Steps/Second: 11,970.1289
Consumption Steps/Second: 32,762.0430
Overall Steps/Second: 8,766.9756

Collection Time: 3.5929
 - Inference Time: 0.1316
 - Env Step Time: 3.3598
Consumption Time: 1.3127
 - GAE Time: 8.773550e-04
 - PPO Learn Time: 1.1852
Collected Timesteps: 43,008
Total Timesteps: 279,269,888
Total Iterations: 5,096














In [ ]:
# cell - 12
# STATUS: drive mounted? binary running? checkpoints present?
import os, subprocess, glob

print("drive mounted:", os.path.isdir("/content/drive"))
if os.path.isdir("/content/drive"):
    ck = "/content/drive/MyDrive/Carnage-V1/checkpoints"
    print("drive ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(ck + "/*")) if os.path.isdir(ck) else "none yet")

LOCAL = "/content/Carnage-V1/build/checkpoints"
print("local ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(LOCAL + "/*")) if os.path.isdir(LOCAL) else "none yet")

r = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("Carnage processes:", r.stdout.strip() if r.stdout.strip() else "none")

In [ ]:
# cell - 13
# STOP training cleanly: SIGTERM (binary saves a checkpoint then exits), SIGKILL fallback.
import subprocess, time
r = subprocess.run(["pkill", "-TERM", "-f", "Carnage"], capture_output=True, text=True)
print("SIGTERM sent:", r.returncode == 0)
time.sleep(20)
r2 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
alive = r2.stdout.strip()
if alive:
    print("still alive, force killing:", alive)
    subprocess.run(["pkill", "-9", "-f", "Carnage"], capture_output=True)
    time.sleep(3)
r3 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("remaining:", r3.stdout.strip() if r3.stdout.strip() else "none")